# Exp 2 — Graph Fusion (alpha schemes)

<a href="https://colab.research.google.com/github/zixian0821-zoe/EN553744-final-project/blob/main/exp2_fusion/02_fusion.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Course:** EN.553.744 Data Science for Large-Scale Graphs (Spring 2026, JHU AMS)
**Team:** Yunwei Chai · Yang Song · Zixian Zhou

## What this notebook shows

Given a music source graph $S_{\text{src}}$ (Exp 1) and a book target graph $S_{\text{tgt}}$, we train a 2-layer GCN under different fusion schemes:

$$S_{\text{fused}} = \alpha\, S_{\text{src}} + (1 - \alpha)\, S_{\text{tgt}}$$

**Schemes compared.** No-graph (MLP) | Source-only ($\alpha = 1$) | Target-only ($\alpha = 0$) | Fixed fused ($\alpha = 0.5$) | Best fixed-$\alpha$ from sweep | Learned $\alpha$ (single scalar parameter) | Per-user $\alpha$ (one scalar per user).

**Headline.** Fixed $\alpha = 0.5$ gives **NDCG@20 = 0.001904** ( **+15.7% vs MLP** ); learned $\alpha$ converges to $\alpha^* \approx 0.478$, confirming roughly equal fusion of source and target.

## Why this notebook only loads results (no re-train)

Each model takes 5–15 minutes to train on CPU (cf. `Train_Time_Total_Seconds` field). We pre-train, save metrics to JSON/CSV, and just load + print here. Re-train code is in `pipeline/run_recommendation_experiment.py` and `pipeline/train_recommendation.py`.

## 0. Setup

In [ ]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS_DIR = '/content/drive/MyDrive/EN553744_final_project/results/exp2_fusion'
else:
    RESULTS_DIR = os.path.abspath('../../drive_upload/results/exp2_fusion')

FIG_DIR = os.path.join(RESULTS_DIR, 'figures')

print('IN_COLAB    :', IN_COLAB)
print('RESULTS_DIR :', RESULTS_DIR)
print('exists      :', os.path.isdir(RESULTS_DIR))
if os.path.isdir(RESULTS_DIR):
    print('contents    :', sorted(os.listdir(RESULTS_DIR)))

## 1. Load result files

In [ ]:
def load_json(name):
    p = os.path.join(RESULTS_DIR, name)
    if not os.path.isfile(p):
        print(f'[missing] {name}')
        return None
    with open(p) as f:
        d = json.load(f)
    print(f'[loaded ] {name}')
    return d

model_comp     = load_json('model_comparison.json')                   # 7 models, baseline run
alpha_sweep    = load_json('exp2_alpha_sweep_results.json')           # patience-80 alpha sweep + conclusion
alpha_sens     = load_json('fused_alpha_sensitivity.json')            # alpha in {0.1, 0.3, 0.5, 0.7, 0.9}
learned_alpha  = load_json('learned_alpha_metrics.json')              # single scalar learned
per_user_alpha = load_json('per_user_alpha_metrics.json')             # one scalar per user
graph_stats    = load_json('graph_statistics.json')                   # source / target graph stats

## 2. Source / target graph statistics

In [ ]:
if graph_stats:
    print('=' * 55)
    print('GRAPH STATISTICS')
    print('=' * 55)
    print(json.dumps(graph_stats, indent=2)[:2000])

## 3. Full model comparison (baseline run)

All 7 schemes on the same train/val/test splits.

In [ ]:
if model_comp:
    df = pd.DataFrame(model_comp)
    cols = ['model', 'graph', 'alpha_value', 'NDCG@20', 'Recall@20',
            'HitRate@20', 'Best_Epoch', 'Train_Time_Total_Seconds']
    df_show = df[cols].copy()
    df_show = df_show.sort_values('NDCG@20', ascending=False).reset_index(drop=True)
    df_show['NDCG@20']    = df_show['NDCG@20'].round(6)
    df_show['Recall@20']  = df_show['Recall@20'].round(6)
    df_show['HitRate@20'] = df_show['HitRate@20'].round(6)
    df_show['Train_min']  = (df_show['Train_Time_Total_Seconds'] / 60).round(1)
    df_show = df_show.drop(columns='Train_Time_Total_Seconds')

    print('MODEL COMPARISON  (sorted by test NDCG@20)')
    print('-' * 95)
    print(df_show.to_string(index=False))

    # headline lifts
    base = df.set_index('model')
    fused = base.loc['Fixed fused GCN', 'NDCG@20']
    nogr  = base.loc['No-graph', 'NDCG@20']
    tgt   = base.loc['Target-only GCN', 'NDCG@20']
    src   = base.loc['Source-only GCN', 'NDCG@20']
    print()
    print(f'Fixed fused (alpha=0.5) NDCG@20 = {fused:.6f}')
    print(f'  vs No-graph       : {(fused - nogr) / nogr * 100:+.1f}%')
    print(f'  vs Target-only    : {(fused - tgt) / tgt  * 100:+.1f}%')
    print(f'  vs Source-only    : {(fused - src) / src  * 100:+.1f}%')

## 4. Patience-80 alpha sweep (the version cited in the report)

Re-run with `patience=80` for fairer convergence. This is the canonical version in the final report.

In [ ]:
if alpha_sweep:
    print('=' * 70)
    print('PATIENCE-80 ALPHA SWEEP')
    print('=' * 70)
    print(f"Experiment : {alpha_sweep['experiment']}")
    print(f"Patience   : {alpha_sweep['patience']}")
    print(f"Seed       : {alpha_sweep['seed']}")
    print(f"Metric     : {alpha_sweep['metric']}")
    print()

    rows = []
    for run, m in alpha_sweep['results'].items():
        rows.append({'run': run, 'alpha': m['alpha'],
                     'val_NDCG@20': m['val_NDCG@20'],
                     'test_NDCG@20': m['test_NDCG@20']})
    sweep_df = pd.DataFrame(rows).sort_values('test_NDCG@20', ascending=False).reset_index(drop=True)
    print(sweep_df.to_string(index=False))

    print()
    print(f"Best alpha             : {alpha_sweep['best_alpha']}")
    print(f"Best test NDCG@20      : {alpha_sweep['best_test_NDCG@20']}")
    print(f"Lift over no-graph     : {alpha_sweep['fusion_lift_over_no_graph']}")
    print(f"Lift over target-only  : {alpha_sweep['fusion_lift_over_target_only']}")
    print()
    print('Conclusion:')
    print(f"  {alpha_sweep['conclusion']}")

## 5. Plot — test NDCG@20 vs alpha (patience-80 sweep)

In [ ]:
if alpha_sweep:
    rows = []
    for run, m in alpha_sweep['results'].items():
        if m['alpha'] is not None and 'fixed_alpha_' in run:
            rows.append({'alpha': m['alpha'],
                         'val_NDCG@20':  m['val_NDCG@20'],
                         'test_NDCG@20': m['test_NDCG@20']})
    df_plot = pd.DataFrame(rows).sort_values('alpha').reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(df_plot['alpha'], df_plot['val_NDCG@20'],  'o-', label='val  NDCG@20', color='steelblue')
    ax.plot(df_plot['alpha'], df_plot['test_NDCG@20'], 's-', label='test NDCG@20', color='lightcoral')

    no_graph_test = alpha_sweep['results']['no_graph_p80']['test_NDCG@20']
    ax.axhline(no_graph_test, color='gray', linestyle='--', linewidth=1, label=f'No-graph baseline ({no_graph_test:.6f})')

    best_alpha = alpha_sweep['best_alpha']
    best_test  = alpha_sweep['best_test_NDCG@20']
    ax.axvline(best_alpha, color='black', linestyle=':', alpha=0.5)
    ax.annotate(f'best alpha = {best_alpha}\ntest = {best_test:.6f}',
                xy=(best_alpha, best_test), xytext=(best_alpha + 0.05, best_test + 0.00005),
                fontsize=10)

    ax.set_xlabel('alpha (source weight)')
    ax.set_ylabel('NDCG@20')
    ax.set_title('Exp 2 — fixed-alpha sweep (patience=80)')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## 6. Learned-alpha and per-user-alpha details

In [ ]:
if learned_alpha:
    print('=' * 60)
    print('LEARNED-ALPHA GCN  (single scalar parameter)')
    print('=' * 60)
    print(f"  alpha*               : {learned_alpha['alpha_value']:.6f}")
    print(f"  best epoch           : {learned_alpha['best_epoch']}  / {learned_alpha['epochs_ran']} ran")
    print(f"  test NDCG@20         : {learned_alpha['test_NDCG@20']:.6f}")
    print(f"  test Recall@20       : {learned_alpha['test_Recall@20']:.6f}")
    print(f"  test HitRate@20      : {learned_alpha['test_HitRate@20']:.6f}")
    print(f"  train time (s)       : {learned_alpha['train_time_total_seconds']:.1f}")
    print(f"  -> alpha* approximately 0.5 confirms balanced fusion")

if per_user_alpha:
    print()
    print('=' * 60)
    print('PER-USER-ALPHA GCN  (one scalar per user)')
    print('=' * 60)
    print(f"  alpha distribution   : mean={per_user_alpha['alpha_mean']:.4f}, "
          f"median={per_user_alpha['alpha_median']:.4f}, std={per_user_alpha['alpha_std']:.4f}")
    print(f"                         min ={per_user_alpha['alpha_min']:.4f}, "
          f"max  ={per_user_alpha['alpha_max']:.4f}")
    print(f"  test NDCG@20         : {per_user_alpha['test_NDCG@20']:.6f}")
    print(f"  test Recall@20       : {per_user_alpha['test_Recall@20']:.6f}")
    print(f"  test HitRate@20      : {per_user_alpha['test_HitRate@20']:.6f}")
    print(f"  -> per-user alpha is essentially constant (std={per_user_alpha['alpha_std']:.4f}),")
    print(f"     so the extra capacity does not help on this dataset.")

## 7. Alpha-sensitivity (CE / KL / RMSE per alpha)

An auxiliary smoothing-only diagnostic: for each fixed alpha, how well does the smoothed signal predict held-out interactions in cosine / KL / RMSE / MAE?

In [ ]:
if alpha_sens:
    df = pd.DataFrame(alpha_sens)
    cols = ['alpha', 'CosSim', 'Top-3', 'KL', 'RMSE', 'MAE', 'Test_CE']
    print('ALPHA SENSITIVITY  (smoothing-only diagnostic, no GCN training)')
    print('-' * 80)
    print(df[cols].to_string(index=False))
    print()
    best_kl    = df.loc[df['KL'].idxmin()]
    best_cos   = df.loc[df['CosSim'].idxmax()]
    print(f"  Best alpha by KL     : {best_kl['alpha']}  (KL = {best_kl['KL']:.4f})")
    print(f"  Best alpha by CosSim : {best_cos['alpha']}  (CosSim = {best_cos['CosSim']:.4f})")
    print('  -> diagnostic prefers alpha = 0.9 (more source weight); GCN training prefers alpha approx 0.5.')
    print('     The two disagree because GCN learns from supervision, not just signal smoothness.')

## 8. Pre-rendered figures

In [ ]:
for fig_name in ['fig_exp2_alpha_sweep.png',
                 'fig_exp2_best_epoch_vs_alpha.png',
                 'fig_exp2_val_vs_test.png']:
    p = os.path.join(FIG_DIR, fig_name)
    if os.path.isfile(p):
        print(f'>> {fig_name}')
        display(Image(p))
    else:
        print(f'[missing] {fig_name}')

## 9. Summary block (paste into report)

In [ ]:
if alpha_sweep and learned_alpha and per_user_alpha:
    print('=' * 65)
    print('SUMMARY FOR REPORT  (Exp 2 — Graph Fusion)')
    print('=' * 65)
    res = alpha_sweep['results']
    print(f"""
Setup
  Models       : 7 schemes (no-graph, source-only, target-only,
                 fixed alpha in {{0.1, 0.3, 0.5, 0.7, 0.9}},
                 learned alpha, per-user alpha)
  GCN          : 2 layers, 30-dim user features (top-30 music genre profile)
  Loss         : BPR (pairwise), patience = {alpha_sweep['patience']}
  Metric       : test NDCG@20
  Seed         : {alpha_sweep['seed']}

Headline (patience-80 sweep)
  Fixed alpha = 0.5            : {res['fixed_alpha_0.5_p80']['test_NDCG@20']:.6f}
  Best alpha ({alpha_sweep['best_alpha']})              : {alpha_sweep['best_test_NDCG@20']:.6f}
  No-graph baseline            : {res['no_graph_p80']['test_NDCG@20']:.6f}
  Target-only                  : {res['target_only_p80']['test_NDCG@20']:.6f}
  Source-only                  : {res['source_only_p80']['test_NDCG@20']:.6f}
  Lift over no-graph           : {alpha_sweep['fusion_lift_over_no_graph']}
  Lift over target-only        : {alpha_sweep['fusion_lift_over_target_only']}

Adaptive alpha
  Learned alpha*               : {learned_alpha['alpha_value']:.4f}    (test NDCG@20 = {learned_alpha['test_NDCG@20']:.6f})
  Per-user alpha (mean+/-std)  : {per_user_alpha['alpha_mean']:.4f} +/- {per_user_alpha['alpha_std']:.4f}
                                 (test NDCG@20 = {per_user_alpha['test_NDCG@20']:.6f})

Conclusion
  Fusion of music + book topology beats either single graph or no graph at all.
  The optimal alpha is approximately 0.5: the supervised model wants
  balanced contributions from both domains. Learned alpha confirms this
  data-drivenly (alpha* approx 0.48). Per-user alpha collapses to a near-constant
  ({per_user_alpha['alpha_std']:.4f} std), so user-level adaptation does not help
  on this dataset at this scale. The NDCG surface across alpha is very flat
  (range {alpha_sweep['fusion_lift_over_target_only']} between best and worst alpha),
  consistent with the small spectral distance W_1(S_src, S_tgt) measured in Exp 4b.
""")